In [1]:
import pandas as pd

In [2]:
df_atc = pd.read_csv('../data/aware_2019_atc.csv')

In [4]:
df_atc

,ATC name,ATC code,route,England AWaRe
0,AMIKACIN,J01GB06,P,Watch
1,AMOXICILLIN,J01CA04,P,Access
2,AMOXICILLIN,J01CA04,O,Access
3,AMOXICILLIN AND ENZYME INHIBITOR,J01CR02,O,Watch
4,AMOXICILLIN AND ENZYME INHIBITOR,J01CR02,P,Watch
...,...,...,...,...
203,TROLEANDOMYCIN,J01FA08,O,Watch
204,TROVAFLOXACIN,J01MA13,O,Watch
205,TROVAFLOXACIN,J01MA13,P,Watch
206,VANCOMYCIN,J01XA01,P,Watch


In [21]:
# Set URL to scrape
url = "https://www.gov.uk/government/publications/uk-aware-antibiotic-classification/uk-access-watch-reserve-and-other-classification-for-antibiotics-uk-aware-antibiotic-classification"

# Read the first HTML table from the page into a DataFrame using the HTML content
df_new = pd.read_html(url)[0]

df_new['Antibiotic'] = df_new['Antibiotic'].str.replace(" #", "", regex=False)

route_list = ["oral", "intravenous"]
for route in route_list:
    df_new['Antibiotic'] = df_new['Antibiotic'].str.replace(fr'(, | ){route}$', '', regex=True)
    
    
name_dict = {
    "Amoxicillin/ clavulanic-acid": "AMOXICILLIN AND ENZYME INHIBITOR",
    "Benzathine-benzylpenicillin": "BENZATHINE BENZYLPENICILLIN",
    "Ceftaroline-fosamil": "CEFTAROLINE_FOSAMIL",
    "Ceftazidime/ avibactam": "CEFTAZIDIME AND BETA-LACTAMASE INHIBITOR",
    "Ceftobiprole-medocaril": "CEFTOBIPROLE",
    "Fusidic acid": "FUSIDIC_ACID",
    "Piperacillin/tazobactam": "PIPERACILLIN AND ENZYME INHIBITOR",
    "Procaine-benzylpenicillin": "PROCAINE BENZYLPENICILLIN",
    "Sulfamethoxazole/ trimethoprim": "CO-TRIMOXAZOLE"
}
df_new['Antibiotic'] = df_new['Antibiotic'].replace(name_dict)



# Display the DataFrame
df_new


,Antibiotic,England-adapted 2019 AWaRe category,UK-adapted 2024 AWaRe category
0,Amikacin,Watch,Watch
1,Amoxicillin,Access,Access
2,AMOXICILLIN AND ENZYME INHIBITOR,Watch,Watch
3,Ampicillin,Access,Access
4,Azithromycin,Watch,Watch
...,...,...,...
85,Tigecycline,Reserve,Reserve
86,Tinidazole,Other,Access
87,Tobramycin,Watch,Watch
88,Trimethoprim,Access,Access


In [22]:
# Create temporary lowercase columns for case-insensitive join
df_atc['atc_lower'] = df_atc['ATC name'].str.lower()
df_new['antibiotic_lower'] = df_new['Antibiotic'].str.lower()

# Perform the inner join on the temporary columns
df_joined = df_atc.merge(df_new, left_on='atc_lower', right_on='antibiotic_lower', how='outer')

# Optionally, drop the helper columns
df_joined = df_joined.drop(columns=['atc_lower', 'antibiotic_lower'])

In [23]:
df_joined

,ATC name,ATC code,route,England AWaRe,Antibiotic,England-adapted 2019 AWaRe category,UK-adapted 2024 AWaRe category
0,AMIKACIN,J01GB06,P,Watch,Amikacin,Watch,Watch
1,AMOXICILLIN,J01CA04,P,Access,Amoxicillin,Access,Access
2,AMOXICILLIN,J01CA04,O,Access,Amoxicillin,Access,Access
3,AMOXICILLIN AND ENZYME INHIBITOR,J01CR02,O,Watch,AMOXICILLIN AND ENZYME INHIBITOR,Watch,Watch
4,AMOXICILLIN AND ENZYME INHIBITOR,J01CR02,P,Watch,AMOXICILLIN AND ENZYME INHIBITOR,Watch,Watch
...,...,...,...,...,...,...,...
220,NaN,NaN,NaN,NaN,Imipenem/cilastatin,Reserve,Reserve
221,NaN,NaN,NaN,NaN,Imipenem/cilastatin/relebactam,Reserve,Reserve
222,NaN,NaN,NaN,NaN,Meropenem/ vaborbactam,Reserve,Reserve
223,NaN,NaN,NaN,NaN,Methenamine,Other,Other


In [24]:
df_joined[df_joined.isnull().any(axis=1)]

,ATC name,ATC code,route,England AWaRe,Antibiotic,England-adapted 2019 AWaRe category,UK-adapted 2024 AWaRe category
7,AMPICILLIN COMBINATIONS,J01CA51,P,Access,NaN,NaN,NaN
12,BEDAQUILINE,J04AK05,O,Other,NaN,NaN,NaN
15,BIAPENEM,J01DH05,P,Watch,NaN,NaN,NaN
16,CAPREOMYCIN,J04AB30,P,Other,NaN,NaN,NaN
17,CARUMONAM,J01DF02,P,Other,NaN,NaN,NaN
...,...,...,...,...,...,...,...
220,NaN,NaN,NaN,NaN,Imipenem/cilastatin,Reserve,Reserve
221,NaN,NaN,NaN,NaN,Imipenem/cilastatin/relebactam,Reserve,Reserve
222,NaN,NaN,NaN,NaN,Meropenem/ vaborbactam,Reserve,Reserve
223,NaN,NaN,NaN,NaN,Methenamine,Other,Other


In [25]:
with pd.option_context('display.max_rows', None):
    display(df_joined[df_joined.isnull().any(axis=1)])

,ATC name,ATC code,route,England AWaRe,Antibiotic,England-adapted 2019 AWaRe category,UK-adapted 2024 AWaRe category
7,AMPICILLIN COMBINATIONS,J01CA51,P,Access,NaN,NaN,NaN
12,BEDAQUILINE,J04AK05,O,Other,NaN,NaN,NaN
15,BIAPENEM,J01DH05,P,Watch,NaN,NaN,NaN
16,CAPREOMYCIN,J04AB30,P,Other,NaN,NaN,NaN
17,CARUMONAM,J01DF02,P,Other,NaN,NaN,NaN
23,CEFCAPENE,J01DD17,O,Watch,NaN,NaN,NaN
24,CEFDINIR,J01DD15,O,Watch,NaN,NaN,NaN
25,CEFDITOREN,J01DD16,O,Watch,NaN,NaN,NaN
27,CEFETAMET,J01DD10,O,Watch,NaN,NaN,NaN
29,CEFMENOXIME,J01DD05,P,Watch,NaN,NaN,NaN


In [ ]:
Amoxicillin/ clavulanic-acid: AMOXICILLIN AND ENZYME INHIBITOR
Benzathine-benzylpenicillin: BENZATHINE BENZYLPENICILLIN
Cefalotin
Cefiderocol
Ceftaroline-fosamil: CEFTAROLINE_FOSAMIL
Ceftazidime/ avibactam: CEFTAZIDIME AND BETA-LACTAMASE INHIBITOR
Ceftobiprole-medocaril: CEFTOBIPROLE
Ceftolozane/ tazobactam	
Dalfopristin/ quinupristin	
Delafloxacin
Eravacycline
Fusidic acid: FUSIDIC_ACID
Imipenem/cilastatin
Imipenem/cilastatin/relebactam
Meropenem/ vaborbactam
Methenamine
Nalidixic Acid
Piperacillin/tazobactam: PIPERACILLIN AND ENZYME INHIBITOR
Procaine-benzylpenicillin: PROCAINE BENZYLPENICILLIN
Sulfamethoxazole/ trimethoprim: CO-TRIMOXAZOLE